# Bước 3: Chuẩn Hóa Dữ Liệu & Xuất Tập Huấn Luyện (Data Scaling & Normalization)
Notebook này xử lý bài toán **khử độ lệch biên độ** giữa các loại cảm biến và phương pháp đo (ECG vs PPG).
Chương trình tự động nạp và chuẩn hóa đồng thời **cả 3 tập dữ liệu**:
1. **MIMIC-III** (1,202 mẫu)
2. **PTB-XL** (3,028 mẫu)
3. **Tập Gộp Combined** (4,230 mẫu: MIMIC + PTB-XL)

Chúng ta áp dụng 2 kỹ thuật chuẩn hóa phổ biến:
- **Z-Score Normalization (`StandardScaler`)**: Biến đổi dữ liệu có trung bình = 0 và độ lệch chuẩn = 1.
- **Min-Max Scaling (`MinMaxScaler`)**: Biến đổi các đặc trưng về khoảng [0, 1].

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
import warnings
warnings.filterwarnings('ignore')

# Cấu hình font & giao diện biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (18, 10)

### 1. Nạp đồng thời 3 tập dữ liệu đặc trưng (MIMIC-III, PTB-XL & Combined)

In [ ]:
data_dir_candidates = [
    '../../data/features',
    '../data/features',
    'data/features'
]
data_dir = next((d for d in data_dir_candidates if os.path.exists(d)), None)
if not data_dir:
    raise FileNotFoundError("❌ Không tìm thấy thư mục data/features!")

mimic_path = os.path.join(data_dir, 'mimic_features.csv')
ptbxl_path = os.path.join(data_dir, 'ptbxl_features.csv')

datasets = {}
if os.path.exists(mimic_path):
    df_mimic = pd.read_csv(mimic_path)
    datasets['MIMIC-III'] = df_mimic
    print(f"✅ Đã nạp MIMIC-III ({len(df_mimic)} mẫu) từ: {mimic_path}")

if os.path.exists(ptbxl_path):
    df_ptbxl = pd.read_csv(ptbxl_path)
    datasets['PTB-XL'] = df_ptbxl
    print(f"✅ Đã nạp PTB-XL ({len(df_ptbxl)} mẫu) từ: {ptbxl_path}")

if 'MIMIC-III' in datasets and 'PTB-XL' in datasets:
    cols = df_mimic.columns.tolist()
    df_combined = pd.concat([df_mimic[cols], df_ptbxl[cols]], ignore_index=True)
    datasets['Combined'] = df_combined
    print(f"✅ Đã tạo Tập Gộp Combined ({len(df_combined)} mẫu: MIMIC + PTB-XL)")

for name, df in datasets.items():
    print(f"📌 {name}: Kích thước = {df.shape}")

### 2. Thực hiện Chuẩn hóa Z-Score & Min-Max Scaling cho cả 3 tập

In [ ]:
scaled_results = {}

for name, df in datasets.items():
    X = df.drop(columns=['status'])
    y = df['status']
    feature_cols = X.columns.tolist()
    
    # Z-Score Scaling
    scaler_z = StandardScaler()
    X_z = pd.DataFrame(scaler_z.fit_transform(X), columns=feature_cols)
    df_zscore = pd.concat([X_z, y.reset_index(drop=True)], axis=1)
    
    # Min-Max Scaling
    scaler_mm = MinMaxScaler()
    X_mm = pd.DataFrame(scaler_mm.fit_transform(X), columns=feature_cols)
    df_minmax = pd.concat([X_mm, y.reset_index(drop=True)], axis=1)
    
    scaled_results[name] = {
        'raw': df,
        'zscore': df_zscore,
        'minmax': df_minmax
    }
    print(f"⚡ Đã chuẩn hóa Z-Score & Min-Max cho tập: {name}")

### 3. Trực quan hóa phân bố SDNN trước và sau khi chuẩn hóa của cả 3 tập

In [ ]:
fig, axes = plt.subplots(len(datasets), 3, figsize=(18, 4 * len(datasets)))

for i, (name, res) in enumerate(scaled_results.items()):
    sns.kdeplot(res['raw']['SDNN'], ax=axes[i][0], color='blue', fill=True)
    axes[i][0].set_title(f'{name} - SDNN Gốc (Original)', fontsize=13, fontweight='bold')
    
    sns.kdeplot(res['zscore']['SDNN'], ax=axes[i][1], color='green', fill=True)
    axes[i][1].set_title(f'{name} - Z-Score Scaled', fontsize=13, fontweight='bold')
    
    sns.kdeplot(res['minmax']['SDNN'], ax=axes[i][2], color='orange', fill=True)
    axes[i][2].set_title(f'{name} - Min-Max Scaled', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 4. Xuất kết quả đã chuẩn hóa vào thư mục `data/features/` và `data/processed/`

In [ ]:
output_dir_candidates = ['../../data/processed', '../data/processed', 'data/processed']
output_dir = next((d for d in output_dir_candidates if os.path.exists(os.path.dirname(d))), '../../data/processed')
os.makedirs(output_dir, exist_ok=True)

file_mapping = {
    'MIMIC-III': ('mimic_zscore_scaled.csv', 'mimic_minmax_scaled.csv'),
    'PTB-XL': ('ptbxl_zscore_scaled.csv', 'ptbxl_minmax_scaled.csv'),
    'Combined': ('zscore_scaled.csv', 'minmax_scaled.csv')
}

for name, res in scaled_results.items():
    if name in file_mapping:
        z_name, mm_name = file_mapping[name]
        z_path = os.path.join(output_dir, z_name)
        mm_path = os.path.join(output_dir, mm_name)
        res['zscore'].to_csv(z_path, index=False)
        res['minmax'].to_csv(mm_path, index=False)
        print(f'🎉 [{name}] Z-Score Scaled -> {z_path}')
        print(f'🎉 [{name}] Min-Max Scaled -> {mm_path}')